# ML Models 02 - Baseline vs LightGBM

**Goal:** Compare baseline models (Naive, MA7d, AR1) against LightGBM.
Baseline MAE per tier = **the bar LightGBM must beat**.

**DATA GATE:** This notebook requires at least 8 price snapshots AND a snapshot 7 days
in the future relative to the chosen training date.
Run the first cell to check availability. If conditions aren't met, come back later.

**Order:** Run notebook 01 first.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parents[1]))

In [2]:
import duckdb
import numpy as np
import pandas as pd

In [3]:
DB_PATH = "../../data/gold/cards.duckdb"
conn = duckdb.connect(DB_PATH, read_only=True)

In [4]:
n_snapshots = conn.execute(
    "SELECT COUNT(DISTINCT snapshot_date) FROM gold_price_features"
).fetchone()[0]
print(f"Available snapshots: {n_snapshots}")
if n_snapshots < 8:
    print("WARNING: fewer than 8 snapshots — walk-forward CV will have limited folds.")

Available snapshots: 17


## 1. Baseline Models

**Naive Forecast:** Predicts tomorrow's price = today's price.
This is the floor — if LightGBM doesn't beat Naive, something is wrong.

**Moving Average 7d:** Average of the last 7 days.
**AR1:** First-order autoregression. Uses yesterday's price change as the only predictor.

All three are imported from `src/ml/models/baseline.py`.

In [5]:
from src.ml.models.baseline import NaiveForecast, MovingAverageForecast, AR1Forecast
from src.ml.features.lag import build_lag_features, build_target
from src.ml.features.pipeline import (
    build_feature_pipeline,
    prepare_training_data,
    get_feature_names,
)
from src.ml.evaluation.metrics import mae, evaluate_per_tier

# Find the latest snapshot_date where snapshot_date + 7 also exists in gold_price_features.
# Using dates[-1] (the most recent snapshot) would fail: build_target needs t+7 to exist,
# and the latest snapshot never has a future row yet.
row = conn.execute("""
    SELECT MAX(t0.snapshot_date)
    FROM gold_price_features t0
    WHERE EXISTS (
        SELECT 1 FROM gold_price_features t7
        WHERE t7.snapshot_date = CAST(t0.snapshot_date AS DATE) + INTERVAL 7 DAY
    )
""").fetchone()[0]

if row is None:
    latest = conn.execute(
        "SELECT MAX(snapshot_date) FROM gold_price_features"
    ).fetchone()[0]
    raise RuntimeError(
        f"DATA GATE: no snapshot with a t+7 counterpart yet. "
        f"Latest snapshot: {latest}. "
        "Training requires two snapshots separated by exactly 7 days. "
        "Come back on 2026-06-18."
    )

SNAPSHOT_DATE = str(row)
print(f"Snapshot date (t+7 exists): {SNAPSHOT_DATE}")

Snapshot date (t+7 exists): 2026-06-05


In [6]:
lag_df = build_lag_features(conn, SNAPSHOT_DATE)
target_df = build_target(conn, SNAPSHOT_DATE)
card_df = conn.execute("SELECT * FROM gold_card_features").df()

# build_lag_features does not compute these columns — baseline models need them
lag_df["log_eur"] = np.log1p(lag_df["eur"])
lag_df["rolling_mean_7d"] = np.log1p(lag_df["rolling_mean_7d"])
lag_df["lag_1d_return"] = (lag_df["eur"] - lag_df["lag_1d"]) / lag_df["lag_1d"].replace(
    0, np.nan
)

# gold_card_features does not yet contain all columns the pipeline expects.
# rarity_ord: ordinal encoding of rarity (pipeline needs a numeric representation).
rarity_map = {"common": 0, "uncommon": 1, "rare": 2, "mythic": 3}
card_df["rarity_ord"] = card_df["rarity"].map(rarity_map)

# is_legendary, has_mtgjson_data: not yet in gold ETL — use safe defaults.
# All rows in gold_card_features have a MTGJson UUID, so has_mtgjson_data=True is correct.
# is_legendary requires type_line parsing, not yet implemented — False is conservative.
card_df["is_legendary"] = False
card_df["has_mtgjson_data"] = True

# top8_appearances_30d, deck_pct: tournament and EDHREC deck data not yet ingested.
card_df["top8_appearances_30d"] = 0.0
card_df["deck_pct"] = 0.0

# Full dataset: lag features + card attributes + target (inner join on uuid)
X_full, y_full = prepare_training_data(lag_df, card_df, target_df)

# Drop rows where the target is NaN (cards with no EUR price at snapshot_date or at t+7).
valid = y_full.notna()
X_full = X_full[valid].reset_index(drop=True)
y_full = y_full[valid].reset_index(drop=True)

# DuckDB returns BOOLEAN columns with NULLs as pandas nullable BooleanDtype.
# When sklearn's ColumnTransformer hstacks these with float arrays, one object
# input makes the entire output object-dtype — which LightGBM rejects.
# Normalise all pandas extension types to plain numpy dtypes before the pipeline.
X_full = X_full.copy()
for col in X_full.columns:
    dtype = X_full[col].dtype
    if hasattr(dtype, "numpy_dtype"):
        ndtype = dtype.numpy_dtype
        if ndtype == np.bool_:
            X_full[col] = X_full[col].fillna(False).astype(bool)
        elif np.issubdtype(ndtype, np.integer):
            X_full[col] = X_full[col].fillna(0).astype(ndtype)
        else:
            X_full[col] = X_full[col].astype(float)

print(f"Dataset: {X_full.shape}  (dropped {(~valid).sum()} rows with NaN target)")

if X_full.empty:
    raise RuntimeError(
        "Dataset is empty after dropping NaN targets — check that card_df and target_df are non-empty."
    )

Dataset: (82413, 45)  (dropped 15940 rows with NaN target)


## 2. LightGBM — First Model

**Important:** Use `objective='mae'`, not `'mse'`.
Why: the price distribution has Pareto alpha ≈ 1.303 < 2, which means infinite variance —
MSE would be dominated by a handful of expensive cards and give a misleading signal.

In [7]:
naive = NaiveForecast()
naive.fit(X_full, y_full)
naive_preds = naive.predict(X_full)

ma7 = MovingAverageForecast(window=7)
ma7.fit(X_full, y_full)
ma7_preds = ma7.predict(X_full)

ar1 = AR1Forecast()
ar1.fit(X_full, y_full)
ar1_preds = ar1.predict(X_full)

print(f"Naive MAE:  {mae(y_full.values, naive_preds):.4f}")
print(f"MA7d  MAE:  {mae(y_full.values, ma7_preds):.4f}")
print(f"AR1   MAE:  {mae(y_full.values, ar1_preds):.4f}")

Naive MAE:  0.0000
MA7d  MAE:  0.0000
AR1   MAE:  0.0000


In [8]:
from src.ml.models.lightgbm_model import LightGBMPriceModel, LightGBMParams

split_idx = int(len(X_full) * 0.8)
X_train_raw, X_test_raw = X_full.iloc[:split_idx], X_full.iloc[split_idx:]
y_train, y_test = y_full.iloc[:split_idx], y_full.iloc[split_idx:]

pipeline = build_feature_pipeline()
X_train_t = pipeline.fit_transform(X_train_raw)
X_test_t = pipeline.transform(X_test_raw)
feat_names = get_feature_names(pipeline)

# ColumnTransformer hstacks float imputer outputs with bool passthrough columns.
# numpy upcast to object when bool meets float — cast explicitly to float64 (same
# fix as in app/main.py lifespan).
X_train_df = pd.DataFrame(np.array(X_train_t, dtype=np.float64), columns=feat_names)
X_test_df = pd.DataFrame(np.array(X_test_t, dtype=np.float64), columns=feat_names)

params = LightGBMParams()
model = LightGBMPriceModel(params)
model.fit(X_train_df, y_train.reset_index(drop=True))

lgbm_preds = model.predict(X_test_df)
print(f"LightGBM MAE: {mae(y_test.values, lgbm_preds):.4f}")
print(f"Naive MAE (test): {mae(y_test.values, naive_preds[split_idx:]):.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002265 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2366
[LightGBM] [Info] Number of data points in the train set: 52744, number of used features: 15
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training 

## 3. Per-Tier Comparison

**Why per tier?** T1 (<€100) has thousands of cards with stable, low prices.
T2 (€100–€1000) has far fewer cards but higher price volatility.
A single global MAE can hide failures in the most valuable tier.

Use `evaluate_per_tier()` from `src/ml/evaluation/metrics.py`.

In [9]:
from src.ml.models.tiered import assign_tier

prices_test = X_full.iloc[split_idx:]["eur"]
tiers_test = prices_test.apply(assign_tier).reset_index(drop=True)
y_test_r = y_test.reset_index(drop=True)

results = evaluate_per_tier(
    y_true=y_test_r,
    predictions={
        "Naive": pd.Series(naive_preds[split_idx:]).reset_index(drop=True),
        "MA7d": pd.Series(ma7_preds[split_idx:]).reset_index(drop=True),
        "LightGBM": pd.Series(lgbm_preds),
    },
    tiers=tiers_test,
)
print(results.to_string())

      model  tier  n_cards           mae      mape
0     Naive     1    16344  0.000000e+00  0.000000
1  LightGBM     1    16344  0.000000e+00  0.000000
2      MA7d     1    16344  1.412704e-08  0.000141
3     Naive     2      110  0.000000e+00  0.000000
4  LightGBM     2      110  0.000000e+00  0.000000
5      MA7d     2      110  1.279895e-07  0.001280
6     Naive     3       29  0.000000e+00  0.000000
7  LightGBM     3       29  0.000000e+00  0.000000
8      MA7d     3       29  2.086702e-07  0.002087


## Observations

*(Run all cells first; fill in values from the printed output above)*

- **Available snapshots:** ___
- **SNAPSHOT_DATE used:** ___ (first date where t+7 exists)
- **Dataset size:** ___ rows × ___ features
- **Naive MAE** — Tier 1: ___ | Tier 2: ___
- **MA7d MAE** — Tier 1: ___ | Tier 2: ___
- **AR1 MAE** — Tier 1: ___ | Tier 2: ___
- **LightGBM MAE** — Tier 1: ___ | Tier 2: ___
- **Does LightGBM beat Naive?** — Tier 1: ___ | Tier 2: ___
- **Hardest baseline to beat:** ___
- **Note on lag features:** At SNAPSHOT_DATE = 2026-06-04 all lag columns are NaN (first snapshot).
  AR1 fills with 0 → identical to Naive. Both improve as history accumulates.
- **Conclusions:** ___

## 4. Logowanie modelu do MLflow

Trenujemy model na **wszystkich** dostępnych danych (nie tylko 80%) i logujemy do MLflow.
Po uruchomieniu tej komórki skopiuj `MODEL_RUN_ID` do docker-compose lub zmiennej środowiskowej.

In [10]:
from src.ml.training.tracking import (
    setup_experiment,
    start_run,
    log_params,
    log_metrics,
    log_model,
)

setup_experiment()

# Train final model on ALL available data (not just the 80% train split)
pipeline_final = build_feature_pipeline()
X_all_t = pipeline_final.fit_transform(X_full)
feat_names_final = get_feature_names(pipeline_final)
X_all_df = pd.DataFrame(np.array(X_all_t, dtype=np.float64), columns=feat_names_final)

final_params = LightGBMParams()
model_final = LightGBMPriceModel(final_params)
model_final.fit(X_all_df, y_full.reset_index(drop=True))

with start_run("lightgbm_initial", snapshot_date=SNAPSHOT_DATE) as run:
    log_params(final_params)
    log_metrics(
        {
            "mae_test": float(mae(y_test.values, lgbm_preds)),
            "mae_naive_test": float(mae(y_test.values, naive_preds[split_idx:])),
            "n_training_rows": float(len(X_all_df)),
            "n_snapshots": float(n_snapshots),
        }
    )
    log_model(model_final)
    run_id = run.info.run_id

print("Model zalogowany do MLflow.")
print(f"\nMODEL_RUN_ID = {run_id}")
print("\nUstaw w PowerShell:")
print(f"  $env:MODEL_RUN_ID = '{run_id}'")
print("  docker compose -f docker/docker-compose.yml up --build")

C:\Workspace\AviariumSoftware\aviarium.columbarius\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002194 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2371
[LightGBM] [Info] Number of data points in the train set: 65930, number of used features: 15
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training 

2026/06/12 13:03:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/12 13:03:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Model zalogowany do MLflow.

MODEL_RUN_ID = 0e53aaea1f574c169c08f1dd15cab5e6

Ustaw w PowerShell:
  $env:MODEL_RUN_ID = '0e53aaea1f574c169c08f1dd15cab5e6'
  docker compose -f docker/docker-compose.yml up --build
